In [1]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Đang sử dụng thiết bị: {device}")

C:\Project_AI\hsu-ai-driven-challenge-2026\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang sử dụng thiết bị: cpu


In [2]:
train_path = "../data/processed/train_master_vi.csv"
val_path = "../data/processed/val_master_vi.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)

train_df = train_df.dropna(subset=['prompt'])
val_df = val_df.dropna(subset=['prompt'])

print(f"Số lượng tập train: {len(train_df)}")
print(f"Số lượng tập val: {len(val_df)}")
print(train_df[['prompt', 'label_unsafe']].head(3))

Số lượng tập train: 59172
Số lượng tập val: 14794
                                              prompt  label_unsafe
0  Có thể áp dụng trình đổ bóng phân đoạn cho toà...             0
1  Tại sao các dịch vụ chuyển tiếp dành cho nạn n...             1
2  Dấu chân môi trường hàng năm của ChatGPT là gì...             0


In [3]:
class SafetyDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.texts = df['prompt'].values
        self.labels = df['label_unsafe'].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

model_name = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = SafetyDataset(train_df, tokenizer, max_len=256)
val_dataset = SafetyDataset(val_df, tokenizer, max_len=256)

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

C:\Project_AI\hsu-ai-driven-challenge-2026\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nhav2\.cache\huggingface\hub\models--vinai--phobert-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [4]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Error while downloading from https://huggingface.co/vinai/phobert-base-v2/resolve/refs%2Fpr%2F3/model.safetensors: HTTPSConnectionPool(host='us.aws.cdn.hf.co', port=443): Read timed out.
Trying to resume download...
Error while downloading from https://huggingface.co/vinai/phobert-base-v2/resolve/refs%2Fpr%2F3/model.safetensors: HTTPSConnectionPool(host='us.aws.cdn.hf.co', port=443): Read timed out.
Trying to resume download...
'(ReadTimeoutError("HTTPSConnectionPool(host='us.aws.cdn.hf.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: efebf4ac-331a-4c25-98e1-4f53cf10c002)')' thrown while requesting GET https://huggingface.co/vinai/phobert-base-v2/resolve/refs%2Fpr%2F3/model.safetensors
Retrying in 1s [Retry 1/5].


In [5]:
EPOCHS = 3
lr = 2e-5

optimizer = AdamW(model.parameters(), lr=lr)

def evaluate(model, val_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='macro')
    return f1

In [6]:
best_f1 = 0.0
save_path = "../models/phobert_model.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    val_f1 = evaluate(model, val_loader, device)
    avg_train_loss = total_loss / len(train_loader)

    print(f"\n--- Kết quả Epoch {epoch + 1} ---")
    print(f"Average Train Loss: {avg_train_loss:.4f}")
    print(f"Validation F1-Score: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), save_path)
        print(f"-> Đã lưu model tốt nhất mới với F1-Score: {best_f1:.4f} vào file '{save_path}'")
    print("-" * 30)

print("\nHuấn luyện hoàn tất!")
print(f"Model tốt nhất có F1-Score = {best_f1:.4f} được lưu tại: {save_path}")

Epoch 1/3: 100%|██████████| 3699/3699 [10:29:57<00:00, 10.22s/it, loss=0.00531] 



--- Kết quả Epoch 1 ---
Average Train Loss: 0.2286
Validation F1-Score: 0.9253
-> Đã lưu model tốt nhất mới với F1-Score: 0.9253 vào file '../models/phobert_model.pt'
------------------------------


Epoch 2/3: 100%|██████████| 3699/3699 [8:53:54<00:00,  8.66s/it, loss=0.844]    



--- Kết quả Epoch 2 ---
Average Train Loss: 0.1587
Validation F1-Score: 0.9329
-> Đã lưu model tốt nhất mới với F1-Score: 0.9329 vào file '../models/phobert_model.pt'
------------------------------


Epoch 3/3: 100%|██████████| 3699/3699 [9:42:06<00:00,  9.44s/it, loss=0.212]     



--- Kết quả Epoch 3 ---
Average Train Loss: 0.1270
Validation F1-Score: 0.9328
------------------------------

Huấn luyện hoàn tất!
Model tốt nhất có F1-Score = 0.9329 được lưu tại: ../models/phobert_model.pt
